In [51]:
import pandas as pd

In [52]:
# Configuration

DATA_DIR = "data"

ORDERS_FILE = f"{DATA_DIR}/olist_orders_dataset.csv"
ORDER_ITEMS_FILE = f"{DATA_DIR}/olist_order_items_dataset.csv"
PRODUCTS_FILE = f"{DATA_DIR}/olist_products_dataset.csv"
CATEGORY_TRANSLATION_FILE = (
    f"{DATA_DIR}/product_category_name_translation.csv"
)

N_BINS = 10
TOP_CATEGORY_COUNT = 10

In [53]:
# Load datasets
orders = pd.read_csv(
    ORDERS_FILE,
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ],
)

order_items = pd.read_csv(ORDER_ITEMS_FILE)
products = pd.read_csv(PRODUCTS_FILE)
translation = pd.read_csv(CATEGORY_TRANSLATION_FILE)

In [54]:
# Helper functions

def positive_seconds(series):
    """
    Removes missing, zero and negative durations.
    """
    series = series.dropna()
    return series[series > 0]


def make_quantile_distribution(series, bins=10, decimals=0):
    """
    Approximates a continuous empirical distribution using
    quantile-based bins. Each bin is represented by its median
    and observed count.
    """
    values = series.dropna()

    if values.empty:
        raise ValueError("Cannot construct distribution from empty data.")

    frame = pd.DataFrame({"value": values})

    frame["bin"] = pd.qcut(
        frame["value"],
        q=bins,
        duplicates="drop"
    )

    distribution = (
        frame
        .groupby("bin", observed=True)["value"]
        .agg(["median", "count"])
        .reset_index(drop=True)
    )

    if decimals == 0:
        distribution["median"] = (
            distribution["median"]
            .round()
            .astype(int)
        )
    else:
        distribution["median"] = (
            distribution["median"]
            .round(decimals)
        )

    return list(
        distribution[
            ["median", "count"]
        ].itertuples(index=False, name=None)
    )


def print_distribution(name, distribution):
    print(f"\n{name} = [")
    for value, count in distribution:
        if isinstance(value, str):
            print(f'    ("{value}", {int(count)}),')
        else:
            print(f"    ({value}, {int(count)}),")
    print("]")

In [55]:
# Explore order statuses
print("=== ORDER STATUS COUNTS ===")
print(
    orders["order_status"]
    .value_counts()
)

=== ORDER STATUS COUNTS ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [56]:
# Cancellation rate
cancel_rate = (
    orders["order_status"]
    .eq("canceled")
    .mean()
)

print(
    "\nCancellation rate:",
    f"{cancel_rate:.12f}",
)


Cancellation rate: 0.006285133898


In [57]:
# Join order items to products and English category names
items_with_english = (
    order_items[
        [
            "order_id",
            "product_id",
        ]
    ]
    .merge(
        products[
            [
                "product_id",
                "product_category_name",
            ]
        ],
        on="product_id",
        how="left",
    )
    .merge(
        translation,
        on="product_category_name",
        how="left",
    )
)

In [58]:
# Validate English category translations

untranslated = items_with_english[
    items_with_english[
        "product_category_name"
    ].notna()
    & items_with_english[
        "product_category_name_english"
    ].isna()
]

print(
    "\nUntranslated category rows:",
    len(untranslated),
)


Untranslated category rows: 24


In [59]:
missing_category_names = (
    items_with_english["product_category_name"]
    .isna()
    .sum()
)

print(
    "Rows with missing original category:",
    missing_category_names
)

Rows with missing original category: 1603


In [60]:
# Inter-arrival distribution

purchase_times = (
    orders["order_purchase_timestamp"]
    .dropna()
    .sort_values()
)

inter_arrival_seconds = (
    purchase_times
    .diff()
    .dt.total_seconds()
)

inter_arrival_seconds = positive_seconds(
    inter_arrival_seconds
)

inter_arrival_distribution = make_quantile_distribution(
    inter_arrival_seconds,
    bins=N_BINS
)

In [61]:
# Approval delay
# Purchase -> approval
approval_seconds = (
    orders["order_approved_at"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds()

approval_seconds = positive_seconds(approval_seconds)

approval_distribution = make_quantile_distribution(
    approval_seconds,
    bins=N_BINS
)

In [62]:
# Dispatch delay
# Approval -> carrier handoff
dispatch_seconds = (
    orders["order_delivered_carrier_date"]
    - orders["order_approved_at"]
).dt.total_seconds()

dispatch_seconds= positive_seconds(dispatch_seconds)

dispatch_distribution = make_quantile_distribution(
    dispatch_seconds,
    bins=N_BINS
)

In [63]:
# Delivery delay
# Carrier handoff -> customer delivery
delivery_seconds = (
    orders["order_delivered_customer_date"]
    - orders["order_delivered_carrier_date"]
).dt.total_seconds()

delivery_seconds = positive_seconds(delivery_seconds)

delivery_distribution = make_quantile_distribution(
    delivery_seconds,
    bins=N_BINS
)

In [64]:
# Order value
# Sum item prices per order
order_values = (
    order_items
    .groupby("order_id")["price"]
    .sum()
)

order_values = order_values[
    order_values > 0
]

order_value_distribution = make_quantile_distribution(
    order_values,
    bins=N_BINS,
    decimals=2
)

In [65]:
# Item count distribution
item_counts = (
    order_items
    .groupby("order_id")
    .size()
)

item_count_counts = (
    item_counts
    .value_counts()
    .sort_index()
    .rename_axis("item_count")
    .reset_index(name="count")
)

item_count_distribution = list(
    item_count_counts[
        ["item_count", "count"]
    ].itertuples(
        index=False,
        name=None,
    )
)

print("\n=== ITEM COUNT FREQUENCIES ===")
print(item_count_counts)


=== ITEM COUNT FREQUENCIES ===
    item_count  count
0            1  88863
1            2   7516
2            3   1322
3            4    505
4            5    204
5            6    198
6            7     22
7            8      8
8            9      3
9           10      8
10          11      4
11          12      5
12          13      1
13          14      2
14          15      2
15          20      2
16          21      1


In [66]:
# Product-category distribution
# Uses English category names and preserves observed frequencies

category_counts = (
    items_with_english[
        "product_category_name_english"
    ]
    .dropna()
    .value_counts()
    .head(TOP_CATEGORY_COUNT)
)

category_distribution = list(
    category_counts
    .rename_axis("category")
    .reset_index(name="count")
    [
        ["category", "count"]
    ]
    .itertuples(
        index=False,
        name=None,
    )
)

print("\n=== TOP CATEGORY FREQUENCIES ===")
print(category_counts)


=== TOP CATEGORY FREQUENCIES ===
product_category_name_english
bed_bath_table           11115
health_beauty             9670
sports_leisure            8641
furniture_decor           8334
computers_accessories     7827
housewares                6964
watches_gifts             5991
telephony                 4545
garden_tools              4347
auto                      4235
Name: count, dtype: int64


In [67]:
# Diagnostics

print("\n=== DATASET SUMMARY ===")

print(
    f"Orders: {len(orders):,}"
)

print(
    f"Order items: {len(order_items):,}"
)

print(
    f"Products: {len(products):,}"
)

print(
    "Valid inter-arrival observations:",
    f"{len(inter_arrival_seconds):,}",
)

print(
    "Valid approval-delay observations:",
    f"{len(approval_seconds):,}",
)

print(
    "Valid dispatch-delay observations:",
    f"{len(dispatch_seconds):,}",
)

print(
    "Valid delivery-delay observations:",
    f"{len(delivery_seconds):,}",
)

print(
    "Orders with positive order values:",
    f"{len(order_values):,}",
)

print(
    "Cancellation rate:",
    f"{cancel_rate:.12f}",
)


=== DATASET SUMMARY ===
Orders: 99,441
Order items: 112,650
Products: 32,951
Valid inter-arrival observations: 98,874
Valid approval-delay observations: 97,985
Valid dispatch-delay observations: 96,285
Valid delivery-delay observations: 96,443
Orders with positive order values: 98,666
Cancellation rate: 0.006285133898


In [68]:
# Final output: distributions to copy into Java workload generator

print(
    "\n=== VALUES TO COPY INTO JAVA WORKLOAD GENERATOR ==="
)

print_distribution(
    "INTER_ARRIVAL_DISTRIBUTION",
    inter_arrival_distribution,
)

print_distribution(
    "APPROVAL_DELAY_DISTRIBUTION",
    approval_distribution,
)

print_distribution(
    "DISPATCH_DELAY_DISTRIBUTION",
    dispatch_distribution,
)

print_distribution(
    "DELIVERY_DELAY_DISTRIBUTION",
    delivery_distribution,
)

print_distribution(
    "ORDER_VALUE_DISTRIBUTION",
    order_value_distribution,
)

print_distribution(
    "ITEM_COUNT_DISTRIBUTION",
    item_count_distribution,
)

print_distribution(
    "CATEGORY_DISTRIBUTION",
    category_distribution,
)

print(
    "\nCANCELLATION_RATE = "
    f"{cancel_rate:.12f}"
)


=== VALUES TO COPY INTO JAVA WORKLOAD GENERATOR ===

INTER_ARRIVAL_DISTRIBUTION = [
    (12, 9949),
    (46, 9871),
    (85, 9960),
    (132, 9834),
    (190, 9860),
    (264, 9909),
    (364, 9865),
    (510, 9880),
    (770, 9867),
    (1650, 9879),
]

APPROVAL_DELAY_DISTRIBUTION = [
    (544, 9857),
    (680, 9777),
    (786, 9834),
    (910, 9742),
    (1106, 9789),
    (1529, 9797),
    (4323, 9793),
    (53863, 9799),
    (93212, 9798),
    (175667, 9799),
]

DISPATCH_DELAY_DISTRIBUTION = [
    (27891, 9629),
    (56438, 9628),
    (77893, 9629),
    (99748, 9628),
    (137849, 9629),
    (180655, 9628),
    (242443, 9629),
    (313122, 9628),
    (427794, 9628),
    (705120, 9629),
]

DELIVERY_DELAY_DISTRIBUTION = [
    (97190, 9645),
    (244660, 9644),
    (354486, 9645),
    (469305, 9643),
    (579011, 9645),
    (672492, 9644),
    (804826, 9644),
    (1039468, 9644),
    (1311122, 9644),
    (2091890, 9645),
]

ORDER_VALUE_DISTRIBUTION = [
    (19.0, 9903),
    (30.0, 996